# Full Model Benchmark: DenseNet (BCE / Focal / CLAHE+Focal) + ConvNeXt

Run cells top to bottom. Fill in `CONVNEXT_CHECKPOINT` (and image size / CLAHE flag) in the cell below once ConvNeXt training finishes.

**Section 1** computes AUC, per-class AUC, confidence stats, and IoU for all 4 models.

**Section 2** generates GradCAM visual comparison PNGs with ground-truth boxes annotated where available.

## Section 1 — Numeric Benchmark (AUC, confidence, IoU)

In [ ]:
"""
Full benchmark rundown across all trained models:
DenseNet (BCE / Focal / CLAHE+Focal) + ConvNeXt.

Reports for each model:
  - Mean AUC + per-class AUC
  - Mean prediction confidence, split by positive vs negative ground truth per class
    (this tells you not just "is it ranking correctly" but "how confident/calibrated is it")
  - Mean IoU + per-class IoU (GradCAM localization quality, for the 8 classes with bbox labels)

Fill in CONVNEXT_CHECKPOINT below once your restarted ConvNeXt run finishes.
"""

import torch
import numpy as np
import pandas as pd
import cv2
import sys
import os
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

sys.path.append('D:/cxr-triage')

from src.models.densenet import DenseNetModel
from src.models.convnext import ConvNeXtModel
from src.inference.gradcam import GradCAM
from src.data.transforms import get_val_transforms

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_ROOT = "F:/X ray dataset/Second Version"

LABELS = [
    'Atelectasis', 'Consolidation', 'Infiltration',
    'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis',
    'Effusion', 'Pneumonia', 'Pleural_Thickening',
    'Cardiomegaly', 'Nodule', 'Mass', 'Hernia'
]
LABEL_TO_IDX = {label: idx for idx, label in enumerate(LABELS)}

BBOX_TO_LABEL = {
    'Atelectasis': 'Atelectasis',
    'Cardiomegaly': 'Cardiomegaly',
    'Effusion': 'Effusion',
    'Infiltrate': 'Infiltration',
    'Mass': 'Mass',
    'Nodule': 'Nodule',
    'Pneumonia': 'Pneumonia',
    'Pneumothorax': 'Pneumothorax'
}

# ─── EDIT THIS ONCE CONVNEXT FINISHES TRAINING ──────────────────────────
CONVNEXT_CHECKPOINT = 'D:/cxr-triage/checkpoints/convnext_focal_fixed/best_model.pth'
CONVNEXT_IMAGE_SIZE = 224   # match whatever run_training.py actually used
CONVNEXT_USE_CLAHE = False  # set True if your ConvNeXt run's transforms use CLAHE
# ─────────────────────────────────────────────────────────────────────

CHECKPOINTS = {
    'DenseNet_BCE':         'D:/cxr-triage/checkpoints/densenet_bce_fixed/best_model.pth',
    'DenseNet_Focal':       'D:/cxr-triage/checkpoints/densenet_focal_fixed/best_model.pth',
    'DenseNet_CLAHE_Focal': 'D:/cxr-triage/checkpoints/clahe_320_logits_fix/best_model.pth',
    'ConvNeXt_Focal':       CONVNEXT_CHECKPOINT,
}

MODEL_CONFIGS = {
    'DenseNet_BCE':         {'image_size': 224, 'use_clahe': False, 'arch': 'densenet'},
    'DenseNet_Focal':       {'image_size': 224, 'use_clahe': False, 'arch': 'densenet'},
    'DenseNet_CLAHE_Focal': {'image_size': 320, 'use_clahe': True,  'arch': 'densenet'},
    'ConvNeXt_Focal':       {'image_size': CONVNEXT_IMAGE_SIZE, 'use_clahe': CONVNEXT_USE_CLAHE, 'arch': 'convnext'},
}

ORIGINAL_SIZE = 1024
PERCENTILE = 90


def find_image(image_name, base_root):
    for folder in os.listdir(base_root):
        if folder.startswith('images_'):
            path = os.path.join(base_root, folder, 'images', image_name)
            if os.path.exists(path):
                return path
    return None


def load_model(checkpoint_path, arch):
    """Loads the correct architecture for the given checkpoint."""
    if arch == 'densenet':
        model = DenseNetModel(num_classes=14, pretrained=False).to(DEVICE)
    elif arch == 'convnext':
        model = ConvNeXtModel(num_classes=14, pretrained=False).to(DEVICE)
    else:
        raise ValueError(f"Unknown arch: {arch}")

    ckpt = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    epoch = ckpt.get('epoch', 'unknown')
    best_auc = ckpt.get('best_auc', 'unknown')
    print(f"  Loaded from epoch {epoch} | checkpoint best_auc: {best_auc}")
    return model


def compute_iou(gt_mask, pred_mask):
    intersection = np.logical_and(gt_mask, pred_mask).sum()
    union = np.logical_or(gt_mask, pred_mask).sum()
    return intersection / union if union > 0 else 0.0


test_df = pd.read_csv('D:/cxr-triage/data/processed/test.csv')
bbox_df_raw = pd.read_csv('F:/X ray dataset/Second Version/BBox_List_2017.csv')
bbox_df_raw = bbox_df_raw.rename(columns={'Bbox [x': 'x', 'y': 'y', 'w': 'w', 'h]': 'h'})[
    ['Image Index', 'Finding Label', 'x', 'y', 'w', 'h']
]
bbox_df_raw = bbox_df_raw[bbox_df_raw['Finding Label'].isin(BBOX_TO_LABEL.keys())].reset_index(drop=True)
test_images = set(test_df['Image Index'].values)


def run_benchmark(model_name, checkpoint_path, image_size, use_clahe, arch):
    print(f"\n{'='*60}")
    print(f"Benchmarking: {model_name} (arch={arch}, size={image_size}, clahe={use_clahe})")
    print(f"{'='*60}")

    if not os.path.exists(checkpoint_path):
        print(f"  SKIPPED — checkpoint not found at {checkpoint_path}")
        return None

    model = load_model(checkpoint_path, arch)
    transform = get_val_transforms(image_size=image_size, use_clahe=use_clahe)

    # ── AUC + confidence loop ──────────────────────────────────────────
    all_preds, all_labels = [], []
    auc_errors = 0

    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f"{model_name} AUC"):
        img_path = find_image(row['Image Index'], BASE_ROOT)
        if img_path is None:
            auc_errors += 1
            continue
        try:
            img = Image.open(img_path).convert('RGB')
            img_tensor = transform(img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                logits = model(img_tensor)
                probs = torch.sigmoid(logits).cpu().numpy()[0]

            label_vec = np.zeros(14)
            for finding in str(row['Finding Labels']).split('|'):
                finding = finding.strip()
                if finding in LABEL_TO_IDX:
                    label_vec[LABEL_TO_IDX[finding]] = 1

            all_preds.append(probs)
            all_labels.append(label_vec)
        except Exception as e:
            auc_errors += 1
            print(f"AUC error on {row['Image Index']}: {e}")
            continue

    print(f"Collected {len(all_preds)} predictions ({auc_errors} errors/skipped)")

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    aucs = {}
    confidence_stats = {}
    if len(all_preds) > 0:
        for i, label in enumerate(LABELS):
            if all_labels[:, i].sum() > 0:
                try:
                    aucs[label] = roc_auc_score(all_labels[:, i], all_preds[:, i])
                except Exception:
                    pass

            pos_mask = all_labels[:, i] == 1
            neg_mask = all_labels[:, i] == 0
            confidence_stats[label] = {
                'mean_conf_pos': float(all_preds[pos_mask, i].mean()) if pos_mask.sum() > 0 else float('nan'),
                'mean_conf_neg': float(all_preds[neg_mask, i].mean()) if neg_mask.sum() > 0 else float('nan'),
                'n_pos': int(pos_mask.sum()),
                'n_neg': int(neg_mask.sum()),
            }

    mean_auc = np.mean(list(aucs.values())) if aucs else float('nan')
    print(f"Mean AUC: {mean_auc:.4f}")

    # ── IoU via GradCAM ─────────────────────────────────────────────────
    gradcam = GradCAM(model)

    bbox_df = bbox_df_raw[bbox_df_raw['Image Index'].isin(test_images)].reset_index(drop=True)
    iou_results = {label: [] for label in BBOX_TO_LABEL.keys()}
    iou_errors = 0

    for _, bbox_row in tqdm(bbox_df.iterrows(), total=len(bbox_df), desc=f"{model_name} IoU"):
        image_name = bbox_row['Image Index']
        bbox_label = bbox_row['Finding Label']
        model_label = BBOX_TO_LABEL.get(bbox_label)
        if model_label is None:
            continue
        try:
            img_path = find_image(image_name, BASE_ROOT)
            if img_path is None:
                iou_errors += 1
                continue

            img = Image.open(img_path).convert('RGB')
            assert img.size == (ORIGINAL_SIZE, ORIGINAL_SIZE)

            scale = image_size / ORIGINAL_SIZE
            gx, gy, gw, gh = bbox_row['x'], bbox_row['y'], bbox_row['w'], bbox_row['h']
            gx1, gy1 = gx * scale, gy * scale
            gx2, gy2 = (gx + gw) * scale, (gy + gh) * scale

            gt_mask = np.zeros((image_size, image_size), dtype=bool)
            gt_mask[int(gy1):int(gy2), int(gx1):int(gx2)] = True

            img_tensor = transform(img).unsqueeze(0)
            heatmap = gradcam.generate(img_tensor.clone(), class_idx=LABEL_TO_IDX[model_label])
            heatmap_resized = cv2.resize(heatmap.astype(np.float32), (image_size, image_size))

            thresh = np.percentile(heatmap_resized, PERCENTILE)
            pred_mask = heatmap_resized >= thresh
            iou = compute_iou(gt_mask, pred_mask)
            iou_results[bbox_label].append(iou)
        except Exception as e:
            iou_errors += 1
            print(f"IoU error on {image_name}: {e}")
            continue

    print(f"IoU errors/skipped: {iou_errors}")

    valid_ious = [np.mean(v) for v in iou_results.values() if v]
    mean_iou = np.mean(valid_ious) if valid_ious else float('nan')
    print(f"Mean IoU: {mean_iou:.4f}")

    gradcam.remove_hooks()

    return {
        'model': model_name,
        'mean_auc': mean_auc,
        'mean_iou': mean_iou,
        'per_class_auc': aucs,
        'per_class_iou': {k: np.mean(v) for k, v in iou_results.items() if v},
        'confidence_stats': confidence_stats,
        'n_test_images': len(all_preds),
    }


# ─── Run all models ──────────────────────────────────────────────────────
all_results = {}
for model_name, ckpt_path in CHECKPOINTS.items():
    cfg = MODEL_CONFIGS[model_name]
    result = run_benchmark(model_name, ckpt_path, cfg['image_size'], cfg['use_clahe'], cfg['arch'])
    if result is not None:
        all_results[model_name] = result

# ─── Summary tables ───────────────────────────────────────────────────────
print(f"\n{'='*70}\nFINAL BENCHMARK SUMMARY\n{'='*70}")
print(f"{'Model':<25} {'Mean AUC':>10} {'Mean IoU':>10} {'N Test Imgs':>14}")
for name, r in all_results.items():
    print(f"{name:<25} {r['mean_auc']:>10.4f} {r['mean_iou']:>10.4f} {r['n_test_images']:>14}")

model_names = list(all_results.keys())

print(f"\n{'='*90}\nPER CLASS AUC COMPARISON\n{'='*90}")
header = f"{'Label':<22}" + "".join(f"{name[:14]:>16}" for name in model_names)
print(header)
for label in LABELS:
    row_str = f"{label:<22}"
    for name in model_names:
        val = all_results[name]['per_class_auc'].get(label, float('nan'))
        row_str += f"{val:>16.4f}"
    print(row_str)

print(f"\n{'='*90}\nPER CLASS IoU COMPARISON (GradCAM localization)\n{'='*90}")
header = f"{'Disease':<22}" + "".join(f"{name[:14]:>16}" for name in model_names)
print(header)
for disease in BBOX_TO_LABEL.keys():
    row_str = f"{disease:<22}"
    for name in model_names:
        val = all_results[name]['per_class_iou'].get(disease, float('nan'))
        row_str += f"{val:>16.4f}"
    print(row_str)

print(f"\n{'='*100}\nPER CLASS CONFIDENCE — mean predicted prob when label IS present vs NOT present\n{'='*100}")
for name in model_names:
    print(f"\n--- {name} ---")
    print(f"{'Label':<22} {'Conf(pos)':>10} {'Conf(neg)':>10} {'Gap':>8} {'n_pos':>7} {'n_neg':>7}")
    cs = all_results[name]['confidence_stats']
    for label in LABELS:
        if label in cs:
            s = cs[label]
            gap = s['mean_conf_pos'] - s['mean_conf_neg']
            print(f"{label:<22} {s['mean_conf_pos']:>10.4f} {s['mean_conf_neg']:>10.4f} "
                  f"{gap:>8.4f} {s['n_pos']:>7} {s['n_neg']:>7}")

print("\nDone. A larger Conf(pos) - Conf(neg) gap means better-separated, "
      "well-calibrated confidence for that class — not just correct ranking (AUC) "
      "but a meaningful confidence signal for triage thresholds.")


## Section 2 — Visual Comparison (GradCAM overlays + ground-truth boxes)

In [ ]:
"""
Visual model comparison: for a set of sample X-rays, shows the original image
(with ground-truth bounding box annotated in red, if one exists for that finding)
next to each model's GradCAM heatmap overlay + predicted confidence.

Saves one PNG grid per case into OUTPUT_DIR. Run this AFTER full_benchmark.py's
CONVNEXT_CHECKPOINT path is filled in and ConvNeXt has finished training.

Layout per case:  [ Original + GT box ] [ DenseNet_BCE ] [ DenseNet_Focal ] [ DenseNet_CLAHE_Focal ] [ ConvNeXt_Focal ]
"""

import torch
import numpy as np
import pandas as pd
import cv2
import sys
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

sys.path.append('D:/cxr-triage')

from src.models.densenet import DenseNetModel
from src.models.convnext import ConvNeXtModel
from src.inference.gradcam import GradCAM
from src.data.transforms import get_val_transforms

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_ROOT = "F:/X ray dataset/Second Version"
OUTPUT_DIR = "D:/cxr-triage/reports/gradcam_comparison"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABELS = [
    'Atelectasis', 'Consolidation', 'Infiltration',
    'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis',
    'Effusion', 'Pneumonia', 'Pleural_Thickening',
    'Cardiomegaly', 'Nodule', 'Mass', 'Hernia'
]
LABEL_TO_IDX = {label: idx for idx, label in enumerate(LABELS)}

BBOX_TO_LABEL = {
    'Atelectasis': 'Atelectasis',
    'Cardiomegaly': 'Cardiomegaly',
    'Effusion': 'Effusion',
    'Infiltrate': 'Infiltration',
    'Mass': 'Mass',
    'Nodule': 'Nodule',
    'Pneumonia': 'Pneumonia',
    'Pneumothorax': 'Pneumothorax'
}

# ─── EDIT ONCE CONVNEXT FINISHES TRAINING ──────────────────────────────
CONVNEXT_CHECKPOINT = 'D:/cxr-triage/checkpoints/convnext_focal_fixed/best_model.pth'
CONVNEXT_IMAGE_SIZE = 224
CONVNEXT_USE_CLAHE = False
# ─────────────────────────────────────────────────────────────────────

MODELS = {
    'DenseNet_BCE':         {'ckpt': 'D:/cxr-triage/checkpoints/densenet_bce_fixed/best_model.pth',
                              'image_size': 224, 'use_clahe': False, 'arch': 'densenet'},
    'DenseNet_Focal':       {'ckpt': 'D:/cxr-triage/checkpoints/densenet_focal_fixed/best_model.pth',
                              'image_size': 224, 'use_clahe': False, 'arch': 'densenet'},
    'DenseNet_CLAHE_Focal': {'ckpt': 'D:/cxr-triage/checkpoints/clahe_320_logits_fix/best_model.pth',
                              'image_size': 320, 'use_clahe': True, 'arch': 'densenet'},
    'ConvNeXt_Focal':       {'ckpt': CONVNEXT_CHECKPOINT,
                              'image_size': CONVNEXT_IMAGE_SIZE, 'use_clahe': CONVNEXT_USE_CLAHE, 'arch': 'convnext'},
}

DISPLAY_SIZE = 320       # every image/heatmap gets resized to this for a consistent grid
ORIGINAL_SIZE = 1024
PERCENTILE = 90          # for the pred_mask, kept consistent with the benchmark script
N_CASES_PER_BBOX_CLASS = 2   # how many sample images to pull per bbox-annotated finding
N_CASES_NO_BBOX = 6          # extra cases for findings without bbox ground truth


def find_image(image_name, base_root):
    for folder in os.listdir(base_root):
        if folder.startswith('images_'):
            path = os.path.join(base_root, folder, 'images', image_name)
            if os.path.exists(path):
                return path
    return None


def load_model(checkpoint_path, arch):
    if arch == 'densenet':
        model = DenseNetModel(num_classes=14, pretrained=False).to(DEVICE)
    elif arch == 'convnext':
        model = ConvNeXtModel(num_classes=14, pretrained=False).to(DEVICE)
    else:
        raise ValueError(f"Unknown arch: {arch}")
    ckpt = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model


def overlay_heatmap(img_rgb_uint8, heatmap):
    """img_rgb_uint8: HxWx3 uint8. heatmap: HxW float in [0,1] (already resized to match)."""
    heatmap_uint8 = np.uint8(255 * heatmap)
    colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    colored = cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(img_rgb_uint8, 0.6, colored, 0.4, 0)
    return overlay


def scale_bbox(gx, gy, gw, gh, from_size, to_size):
    scale = to_size / from_size
    return gx * scale, gy * scale, gw * scale, gh * scale


test_df = pd.read_csv('D:/cxr-triage/data/processed/test.csv')
bbox_df_raw = pd.read_csv('F:/X ray dataset/Second Version/BBox_List_2017.csv')
bbox_df_raw = bbox_df_raw.rename(columns={'Bbox [x': 'x', 'y': 'y', 'w': 'w', 'h]': 'h'})[
    ['Image Index', 'Finding Label', 'x', 'y', 'w', 'h']
]
test_images = set(test_df['Image Index'].values)
bbox_df = bbox_df_raw[bbox_df_raw['Image Index'].isin(test_images)].reset_index(drop=True)


def select_cases():
    """Returns list of dicts: {image_name, finding, bbox (or None)}"""
    cases = []

    # Cases WITH ground-truth bbox, spread across the 8 bbox-annotated findings
    for bbox_label in BBOX_TO_LABEL.keys():
        subset = bbox_df[bbox_df['Finding Label'] == bbox_label]
        picks = subset.sample(n=min(N_CASES_PER_BBOX_CLASS, len(subset)), random_state=42) if len(subset) else subset
        for _, row in picks.iterrows():
            cases.append({
                'image_name': row['Image Index'],
                'finding': BBOX_TO_LABEL[bbox_label],
                'bbox': (row['x'], row['y'], row['w'], row['h']),
            })

    # Extra cases WITHOUT bbox ground truth, for findings not covered by BBox_List_2017
    no_bbox_labels = [l for l in LABELS if l not in BBOX_TO_LABEL.values()]
    for label in no_bbox_labels[:N_CASES_NO_BBOX]:
        matches = test_df[test_df['Finding Labels'].astype(str).str.contains(label, na=False)]
        if len(matches) > 0:
            row = matches.sample(n=1, random_state=42).iloc[0]
            cases.append({
                'image_name': row['Image Index'],
                'finding': label,
                'bbox': None,
            })

    return cases


def build_case_figure(case, models_loaded):
    image_name = case['image_name']
    finding = case['finding']
    bbox = case['bbox']
    class_idx = LABEL_TO_IDX[finding]

    img_path = find_image(image_name, BASE_ROOT)
    if img_path is None:
        print(f"  Image not found: {image_name}, skipping case")
        return

    orig_img = Image.open(img_path).convert('RGB')
    orig_resized = orig_img.resize((DISPLAY_SIZE, DISPLAY_SIZE))
    orig_np = np.array(orig_resized)

    n_panels = 1 + len(models_loaded)
    fig, axes = plt.subplots(1, n_panels, figsize=(4.2 * n_panels, 4.6))

    # Panel 0: original + GT box
    axes[0].imshow(orig_np)
    axes[0].set_title(f"Original\n{image_name}\nFinding: {finding}", fontsize=9)
    axes[0].axis('off')
    if bbox is not None:
        gx, gy, gw, gh = scale_bbox(*bbox, from_size=ORIGINAL_SIZE, to_size=DISPLAY_SIZE)
        rect = patches.Rectangle((gx, gy), gw, gh, linewidth=2, edgecolor='red', facecolor='none')
        axes[0].add_patch(rect)
        axes[0].text(gx, max(gy - 5, 0), 'ground truth', color='red', fontsize=8, weight='bold')
    else:
        axes[0].text(5, DISPLAY_SIZE - 10, 'no bbox annotation available', color='orange', fontsize=8)

    # Panels 1..N: each model's heatmap overlay + predicted confidence
    for i, (model_name, bundle) in enumerate(models_loaded.items(), start=1):
        model = bundle['model']
        transform = bundle['transform']
        image_size = bundle['image_size']
        gradcam = bundle['gradcam']

        img_tensor = transform(orig_img).unsqueeze(0)

        with torch.no_grad():
            logits = model(img_tensor.to(DEVICE))
            conf = torch.sigmoid(logits)[0, class_idx].item()

        heatmap = gradcam.generate(img_tensor.clone(), class_idx=class_idx)
        heatmap_resized = cv2.resize(heatmap.astype(np.float32), (DISPLAY_SIZE, DISPLAY_SIZE))
        hmax = heatmap_resized.max()
        heatmap_norm = heatmap_resized / hmax if hmax > 0 else heatmap_resized

        overlay = overlay_heatmap(orig_np, heatmap_norm)
        axes[i].imshow(overlay)
        axes[i].set_title(f"{model_name}\nconfidence: {conf:.3f}", fontsize=9)
        axes[i].axis('off')

        if bbox is not None:
            gx, gy, gw, gh = scale_bbox(*bbox, from_size=ORIGINAL_SIZE, to_size=DISPLAY_SIZE)
            rect = patches.Rectangle((gx, gy), gw, gh, linewidth=2, edgecolor='red', facecolor='none')
            axes[i].add_patch(rect)

    plt.tight_layout()
    safe_finding = finding.replace(' ', '_')
    out_path = os.path.join(OUTPUT_DIR, f"{safe_finding}_{image_name.replace('.png','')}.png")
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved: {out_path}")


def main():
    print("Loading models...")
    models_loaded = {}
    for name, cfg in MODELS.items():
        if not os.path.exists(cfg['ckpt']):
            print(f"  SKIPPING {name} — checkpoint not found at {cfg['ckpt']}")
            continue
        model = load_model(cfg['ckpt'], cfg['arch'])
        transform = get_val_transforms(image_size=cfg['image_size'], use_clahe=cfg['use_clahe'])
        gradcam = GradCAM(model)
        models_loaded[name] = {
            'model': model, 'transform': transform,
            'image_size': cfg['image_size'], 'gradcam': gradcam
        }
        print(f"  Loaded {name}")

    if not models_loaded:
        print("No models loaded — check checkpoint paths.")
        return

    cases = select_cases()
    print(f"\nSelected {len(cases)} cases. Generating comparison figures...")

    for case in cases:
        print(f"\nCase: {case['image_name']} | {case['finding']} | bbox={'yes' if case['bbox'] else 'no'}")
        build_case_figure(case, models_loaded)

    for bundle in models_loaded.values():
        bundle['gradcam'].remove_hooks()

    print(f"\nDone. All comparison figures saved to: {OUTPUT_DIR}")


if __name__ == '__main__':
    main()
